<a href="https://colab.research.google.com/github/ChanderValasai/gender-pay-gap-audit/blob/main/notebooks/04_confounder_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/data-analysis-projects/pakistan-tech-pay-gap'
os.chdir(PROJECT_PATH)

import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('data/processed/cleaned_survey_2025.csv')

Mounted at /content/drive


In [2]:
print(df.groupby('CountryGroup')['YearsCode'].describe())

                 count       mean        std  min   25%   50%   75%    max
CountryGroup                                                              
Pakistan         112.0   8.955357   6.450628  1.0   5.0   7.0  11.0   40.0
Rest of World  23000.0  17.931435  10.930258  1.0  10.0  15.0  25.0  100.0


In [3]:
print(df['YearsCode'].describe())
print("\nRows with YearsCode > 50:", (df['YearsCode'] > 50).sum())
print(df[df['YearsCode'] > 50]['YearsCode'].value_counts())

count    23112.000000
mean        17.887937
std         10.930690
min          1.000000
25%         10.000000
50%         15.000000
75%         25.000000
max        100.000000
Name: YearsCode, dtype: float64

Rows with YearsCode > 50: 76
YearsCode
53.0     11
52.0     11
55.0     10
60.0      7
51.0      6
58.0      5
56.0      5
54.0      5
100.0     5
57.0      4
61.0      2
68.0      1
59.0      1
63.0      1
70.0      1
62.0      1
Name: count, dtype: int64


In [4]:
# Re-merge Age in for a plausibility check (only if not already in df)
high_exp = df[df['YearsCode'] > 50][['YearsCode', 'Age']] if 'Age' in df.columns else "Age not in cleaned df"
print(high_exp)

       YearsCode                Age
130         53.0  65 years or older
308         52.0  65 years or older
675         53.0  65 years or older
687         51.0    55-64 years old
696         58.0  65 years or older
...          ...                ...
21651       60.0  65 years or older
21836       51.0    55-64 years old
21936       62.0  65 years or older
22075       52.0  65 years or older
22493       51.0    55-64 years old

[76 rows x 2 columns]


In [5]:
YEARSCODE_CAP = 50
before = len(df)
df = df[df['YearsCode'] <= YEARSCODE_CAP]
print(f"Rows removed: {before - len(df)} ({round((before-len(df))/before*100,2)}%)")
print(df.groupby('CountryGroup')['YearsCode'].describe())

Rows removed: 76 (0.33%)
                 count       mean        std  min   25%   50%   75%   max
CountryGroup                                                             
Pakistan         112.0   8.955357   6.450628  1.0   5.0   7.0  11.0  40.0
Rest of World  22924.0  17.797069  10.674900  1.0  10.0  15.0  25.0  50.0


In [6]:
# Create experience bands
bins = [0, 5, 10, 15, 25, 100]
labels = ['0-5', '5-10', '10-15', '15-25', '25+']
df['ExpBand'] = pd.cut(df['YearsCode'], bins=bins, labels=labels, include_lowest=True)

# Sample sizes per band per group — check BEFORE looking at salary
print(df.groupby(['ExpBand', 'CountryGroup']).size().unstack())

CountryGroup  Pakistan  Rest of World
ExpBand                              
0-5                 33           1993
5-10                50           5215
10-15               17           4662
15-25                7           6107
25+                  5           4947


/tmp/ipykernel_3102/1194554722.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(['ExpBand', 'CountryGroup']).size().unstack())


In [7]:
strat_summary = df.groupby(['ExpBand', 'CountryGroup'])['ConvertedCompYearly'].agg(['count', 'median', 'mean'])
print(strat_summary)

                       count    median           mean
ExpBand CountryGroup                                 
0-5     Pakistan          33    2950.0    6595.030303
        Rest of World   1993   23203.0   38611.514300
5-10    Pakistan          50   10860.5   16885.000000
        Rest of World   5215   51071.0   64927.760307
10-15   Pakistan          17   16856.0   21573.000000
        Rest of World   4662   75410.0   92877.057915
15-25   Pakistan           7   25284.0   26140.857143
        Rest of World   6107   92812.0  114724.880793
25+     Pakistan           5   18000.0   22932.200000
        Rest of World   4947  111435.0  132740.964019


/tmp/ipykernel_3102/1149597727.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  strat_summary = df.groupby(['ExpBand', 'CountryGroup'])['ConvertedCompYearly'].agg(['count', 'median', 'mean'])


In [9]:
import statsmodels.formula.api as smf

df['LogComp'] = np.log(df['ConvertedCompYearly'])
# Use log(salary) since we established this is better-behaved
model = smf.ols('LogComp ~ YearsCode + CountryGroup', data=df).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                LogComp   R-squared:                       0.167
Model:                            OLS   Adj. R-squared:                  0.167
Method:                 Least Squares   F-statistic:                     2307.
Date:                Thu, 09 Jul 2026   Prob (F-statistic):               0.00
Time:                        15:27:59   Log-Likelihood:                -32618.
No. Observations:               23036   AIC:                         6.524e+04
Df Residuals:                   23033   BIC:                         6.527e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                    coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

In [10]:
import numpy as np
multiplier = np.exp(1.6713)
print(f"Rest of World earns {multiplier:.2f}x Pakistan's salary, holding YearsCode constant")

Rest of World earns 5.32x Pakistan's salary, holding YearsCode constant


In [11]:
model_robust = smf.ols('LogComp ~ YearsCode + CountryGroup', data=df).fit(cov_type='HC3')
print(model_robust.summary())

                            OLS Regression Results                            
Dep. Variable:                LogComp   R-squared:                       0.167
Model:                            OLS   Adj. R-squared:                  0.167
Method:                 Least Squares   F-statistic:                     2195.
Date:                Thu, 09 Jul 2026   Prob (F-statistic):               0.00
Time:                        15:30:38   Log-Likelihood:                -32618.
No. Observations:               23036   AIC:                         6.524e+04
Df Residuals:                   23033   BIC:                         6.527e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                                    coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------
Intercept     

## Confounder-Controlled Analysis Summary

**Method 1 — Stratified comparison:** Pakistan's median compensation is lower than Rest-of-World's
in every experience band tested (0-5, 5-10, 10-15, 15-25, 25+ years). Relative gap ranges from ~3.7x
to ~7.9x depending on band, with some narrowing at mid-career levels. Top two bands (15-25, 25+) have
very small Pakistan samples (n=7, n=5) and should be treated as low-confidence/directional only.

**Method 2 — OLS regression (log salary ~ YearsCode + CountryGroup):**
- CountryGroup coefficient: 1.6713 (p < .001), robust to heteroskedasticity-consistent (HC3) standard errors
- Translated: holding experience constant, Rest-of-World respondents earn approximately 5.3x what
  Pakistan respondents earn
- YearsCode coefficient: 0.0397 (p < .001) — each additional year of experience associates with ~4.05%
  higher salary, holding country group constant
- Model R² = 0.167 — experience and country group together explain only ~17% of salary variance;
  substantial unexplained variation remains (role, seniority, company size, education, remote-work
  status not yet included)

**Conclusion:** The gap identified in the unadjusted analysis (Phase 6) is NOT fully explained by
experience differences between groups. Two independent methods (stratification and regression)
converge on a consistent estimate (~4-8x range across methods), providing robust triangulated evidence.

**What this does NOT establish:**
- Causation — this is a controlled association, not a causal claim. Unmeasured confounders (role,
  seniority, company size, industry, education, remote-work arrangement) have not been ruled out.
- Purchasing-power comparison — this measures nominal USD compensation only; cost-of-living differences
  are not adjusted for and would be expected to explain some portion of any nominal gap.
- Discrimination — "gap not explained by experience" is a narrower and more defensible claim than
  "gap caused by unfair treatment." A stakeholder should not use this analysis to claim proof of
  discrimination.

**Low R² caveat for report:** stated explicitly as a limitation and future-work item, not glossed over.